## Exploratory Data Analysis

### Import Libraries

In [1]:
import sys
import polars as pl
import polars.selectors as pol_sel

### Show Python & Library Versions

In [2]:
l = 8
r = 12

print("Python".rjust(l), ":", sys.version[0:6].ljust(r))
print("Polars".rjust(l), ":", pl.__version__.ljust(r))

  Python : 3.11.4      
  Polars : 1.12.0      


### Load CSV File into Polars DataFrame

In [3]:
df = pl.read_csv(
    "../couchbase_scripts/data/Marketing.csv", 
    ignore_errors=True, 
    truncate_ragged_lines=True
    )

df

CUST_ID,ADVERTISING_INDICATOR,ATTACHMENT_ALLOWED_INDICATOR,PREFERRED_COMMUNICATION_FORM,IMPORTANCE_LEVEL_CODE,INFLUENCE_SCORE,MARKET_GROUP,LOYALTY_RATING_CODE,RECORDED_VOICE_SAMPLE_ID,REFERRALS_VALUE_CODE,RELATIONSHIP_START_DATE
str,bool,bool,str,str,i64,str,i64,i64,str,str
"""CUST-417911""",true,true,"""Email""","""Low priority""",107660,"""Accumulating""",113633,79027,"""Very High""","""7/12/16"""
"""CUST-758898""",true,true,"""Email""","""Low priority""",107664,"""Accumulating""",113633,79172,"""Very High""","""7/12/16"""
"""CUST-958684""",true,true,"""Teleconference""","""High priority""",92114,"""Accumulating""",62773,70208,"""Very Low""","""8/15/17"""
"""CUST-124574""",true,true,"""In Person""","""High priority""",52510,"""Accumulating""",104010,114804,"""Very High""","""8/24/16"""
"""CUST-198781""",true,true,"""In Person""","""High priority""",52562,"""Accumulating""",104010,114786,"""Very High""","""8/24/16"""
…,…,…,…,…,…,…,…,…,…,…
"""CUST-717296""",true,true,"""Other""","""High priority""",72649,"""Accumulating""",54611,60037,"""Low""","""8/28/17"""
"""CUST-780694""",true,true,"""Other""","""High priority""",72584,"""Accumulating""",54611,60168,"""Low""","""8/28/17"""
"""CUST-787420""",true,true,"""Video Conference""","""Normal priority""",131786,"""Gifting""",80883,90987,"""Very High""","""11/29/17"""


### Retrieve Number of Nulls in Each Feature

In [4]:
def count_nulls(df: pl.DataFrame) -> pl.DataFrame:
    return pl.DataFrame({
        "feature": df.columns,
        "null_count": df.null_count().row(0)
    })

pl.Config.set_tbl_rows(35)

null_counts = count_nulls(df)
null_counts

feature,null_count
str,i64
"""CUST_ID""",0
"""ADVERTISING_INDICATOR""",0
"""ATTACHMENT_ALLOWED_INDICATOR""",0
"""PREFERRED_COMMUNICATION_FORM""",0
"""IMPORTANCE_LEVEL_CODE""",0
"""INFLUENCE_SCORE""",0
"""MARKET_GROUP""",0
"""LOYALTY_RATING_CODE""",0
"""RECORDED_VOICE_SAMPLE_ID""",0


### Retrieve Basic Information About DataFrame

In [5]:
def print_schema(df: pl.DataFrame):
    print(f"{'Column':<30} | {'Data Type'}")
    print("-" * 60)
    for name, dtype in zip(df.columns, df.dtypes):
        print(f"{name:<30} | {dtype}")

print_schema(df)

Column                         | Data Type
------------------------------------------------------------
CUST_ID                        | String
ADVERTISING_INDICATOR          | Boolean
ATTACHMENT_ALLOWED_INDICATOR   | Boolean
PREFERRED_COMMUNICATION_FORM   | String
IMPORTANCE_LEVEL_CODE          | String
INFLUENCE_SCORE                | Int64
MARKET_GROUP                   | String
LOYALTY_RATING_CODE            | Int64
RECORDED_VOICE_SAMPLE_ID       | Int64
REFERRALS_VALUE_CODE           | String
RELATIONSHIP_START_DATE        | String


### Display Summary Statistics for All Columns

In [6]:
summary = df.describe()
print(summary)

shape: (9, 12)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ statistic ┆ CUST_ID   ┆ ADVERTISI ┆ ATTACHMEN ┆ … ┆ LOYALTY_R ┆ RECORDED_ ┆ REFERRALS ┆ RELATION │
│ ---       ┆ ---       ┆ NG_INDICA ┆ T_ALLOWED ┆   ┆ ATING_COD ┆ VOICE_SAM ┆ _VALUE_CO ┆ SHIP_STA │
│ str       ┆ str       ┆ TOR       ┆ _INDICATO ┆   ┆ E         ┆ PLE_ID    ┆ DE        ┆ RT_DATE  │
│           ┆           ┆ ---       ┆ R         ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│           ┆           ┆ f64       ┆ ---       ┆   ┆ f64       ┆ f64       ┆ str       ┆ str      │
│           ┆           ┆           ┆ f64       ┆   ┆           ┆           ┆           ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ count     ┆ 2000      ┆ 2000.0    ┆ 2000.0    ┆ … ┆ 2000.0    ┆ 2000.0    ┆ 2000      ┆ 2000     │
│ null_coun ┆ 0         ┆ 0.0       ┆ 0.0       ┆ … ┆ 0.0       ┆ 0.0       

### Find Longest Text Length in Each Column

In [7]:
# Create an empty list to store max lengths for each string column
longest_text_lengths = []

# Loop through the columns to check for string columns
string_columns = [col for col in df.columns if df[col].dtype == pl.Utf8]

max_lengths = {}
for col in string_columns:
    max_length = df.select(pl.col(col).str.len_chars().max()).to_numpy()[0, 0]
    max_lengths[col] = max_length

df_max_lengths = pl.DataFrame(max_lengths)

df_max_lengths

CUST_ID,PREFERRED_COMMUNICATION_FORM,IMPORTANCE_LEVEL_CODE,MARKET_GROUP,REFERRALS_VALUE_CODE,RELATIONSHIP_START_DATE
u32,u32,u32,u32,u32,u32
11,16,15,12,9,8


### Retrieve Data Types of All Columns

In [8]:
print("Column data types:\n", df.dtypes)

Column data types:
 [String, Boolean, Boolean, String, String, Int64, String, Int64, Int64, String, String]


### Count Unique Values in Each Column

In [9]:
all_columns = [col for col in df.columns]

for col in all_columns:
    unique_counts = df[col].n_unique()
    print(f"Unique values in {col} :".rjust(48), f"{unique_counts}".ljust(6))

                      Unique values in CUST_ID : 1999  
        Unique values in ADVERTISING_INDICATOR : 2     
 Unique values in ATTACHMENT_ALLOWED_INDICATOR : 2     
 Unique values in PREFERRED_COMMUNICATION_FORM : 5     
        Unique values in IMPORTANCE_LEVEL_CODE : 3     
              Unique values in INFLUENCE_SCORE : 1969  
                 Unique values in MARKET_GROUP : 4     
          Unique values in LOYALTY_RATING_CODE : 997   
     Unique values in RECORDED_VOICE_SAMPLE_ID : 1978  
         Unique values in REFERRALS_VALUE_CODE : 5     
      Unique values in RELATIONSHIP_START_DATE : 228   


### Check Distribution of Numerical Columns

In [10]:
numerical_cols = [item for item in all_columns if item not in string_columns]
numerical_cols = [item for item in numerical_cols if item not in ['ID']]

numerical_cols

for col in numerical_cols:
    distribution = df.select(col).describe()
    print(col)
    print(distribution, '\n\n')

ADVERTISING_INDICATOR
shape: (9, 2)
┌────────────┬───────────────────────┐
│ statistic  ┆ ADVERTISING_INDICATOR │
│ ---        ┆ ---                   │
│ str        ┆ f64                   │
╞════════════╪═══════════════════════╡
│ count      ┆ 2000.0                │
│ null_count ┆ 0.0                   │
│ mean       ┆ 0.999                 │
│ std        ┆ null                  │
│ min        ┆ 0.0                   │
│ 25%        ┆ null                  │
│ 50%        ┆ null                  │
│ 75%        ┆ null                  │
│ max        ┆ 1.0                   │
└────────────┴───────────────────────┘ 


ATTACHMENT_ALLOWED_INDICATOR
shape: (9, 2)
┌────────────┬──────────────────────────────┐
│ statistic  ┆ ATTACHMENT_ALLOWED_INDICATOR │
│ ---        ┆ ---                          │
│ str        ┆ f64                          │
╞════════════╪══════════════════════════════╡
│ count      ┆ 2000.0                       │
│ null_count ┆ 0.0                          │
│ mean     

### List Unique Values For Certain Features

In [11]:
def list_unique_values_under_threshold(df: pl.DataFrame, threshold: int = 5000):
    for col in df.columns:
        unique_count = df.select(pl.col(col).n_unique()).item()
        if unique_count < threshold:
            unique_vals = df.select(pl.col(col).unique().sort()).to_series()
            print(f"Column: {col} ({unique_count} unique values)")
            print(unique_vals)
            print("-" * 50)

# Apply to your DataFrame
list_unique_values_under_threshold(df)

Column: CUST_ID (1999 unique values)
shape: (1_999,)
Series: 'CUST_ID' [str]
[
	"CUST-100067"
	"CUST-100282"
	"CUST-100396"
	"CUST-100400"
	"CUST-100417"
	"CUST-100655"
	"CUST-100759"
	"CUST-101785"
	"CUST-101865"
	"CUST-102995"
	"CUST-103026"
	"CUST-103108"
	"CUST-103227"
	"CUST-104309"
	"CUST-104713"
	"CUST-105166"
	"CUST-106996"
	"CUST-107207"
	…
	"CUST-993400"
	"CUST-994052"
	"CUST-994211"
	"CUST-994613"
	"CUST-994795"
	"CUST-994918"
	"CUST-994965"
	"CUST-995095"
	"CUST-996204"
	"CUST-997038"
	"CUST-997072"
	"CUST-997260"
	"CUST-997681"
	"CUST-998954"
	"CUST-998988"
	"CUST-999095"
	"CUST-999182"
]
--------------------------------------------------
Column: ADVERTISING_INDICATOR (2 unique values)
shape: (2,)
Series: 'ADVERTISING_INDICATOR' [bool]
[
	false
	true
]
--------------------------------------------------
Column: ATTACHMENT_ALLOWED_INDICATOR (2 unique values)
shape: (2,)
Series: 'ATTACHMENT_ALLOWED_INDICATOR' [bool]
[
	false
	true
]
-------------------------------------------

In [12]:
def list_unique_values_over_threshold(df: pl.DataFrame, threshold: int = 5000):
    for col in df.columns:
        unique_count = df.select(pl.col(col).n_unique()).item()
        if unique_count > threshold:
            unique_vals = df.select(pl.col(col).unique().sort()).to_series()
            print(f"Column: {col} ({unique_count} unique values)")
            print(unique_vals)
            print("-" * 50)

# Apply to your DataFrame
list_unique_values_over_threshold(df)

### How Many Records Remain IF I Remove Records With Any Nulls In It

In [13]:
def drop_rows_with_any_nulls(df: pl.DataFrame) -> pl.DataFrame:
    """
    Removes all rows from a Polars DataFrame that contain any null values.
    """
    return df.drop_nulls()


drop_rows_with_any_nulls(df)

CUST_ID,ADVERTISING_INDICATOR,ATTACHMENT_ALLOWED_INDICATOR,PREFERRED_COMMUNICATION_FORM,IMPORTANCE_LEVEL_CODE,INFLUENCE_SCORE,MARKET_GROUP,LOYALTY_RATING_CODE,RECORDED_VOICE_SAMPLE_ID,REFERRALS_VALUE_CODE,RELATIONSHIP_START_DATE
str,bool,bool,str,str,i64,str,i64,i64,str,str
"""CUST-417911""",true,true,"""Email""","""Low priority""",107660,"""Accumulating""",113633,79027,"""Very High""","""7/12/16"""
"""CUST-758898""",true,true,"""Email""","""Low priority""",107664,"""Accumulating""",113633,79172,"""Very High""","""7/12/16"""
"""CUST-958684""",true,true,"""Teleconference""","""High priority""",92114,"""Accumulating""",62773,70208,"""Very Low""","""8/15/17"""
"""CUST-124574""",true,true,"""In Person""","""High priority""",52510,"""Accumulating""",104010,114804,"""Very High""","""8/24/16"""
"""CUST-198781""",true,true,"""In Person""","""High priority""",52562,"""Accumulating""",104010,114786,"""Very High""","""8/24/16"""
"""CUST-228676""",true,true,"""In Person""","""Normal priority""",88442,"""Spending""",82003,144401,"""Very High""","""4/6/17"""
"""CUST-464124""",true,true,"""Teleconference""","""High priority""",119249,"""Accumulating""",134016,50803,"""Low""","""2/10/16"""
"""CUST-529407""",true,false,"""Teleconference""","""High priority""",119301,"""Accumulating""",134016,50855,"""Low""","""2/10/16"""
"""CUST-523002""",true,true,"""Email""","""Low priority""",67635,"""Accumulating""",47989,117630,"""Very High""","""2/13/16"""
